# Updating a Model After Change

## Prediction Before Observation

For each evaluation date \(t\), the model available after date \(t-1\) produces

$$
\widehat y_t=f_{t-1}(x_t),
$$

where \(x_t\) contains only lagged returns and trailing summaries known through date \(t-1\). After \(y_t\) is observed, the mistake is

$$
L_t=\mathbf{1}\{\widehat y_t\ne y_t\}.
$$

Here, \(f_{t-1}\) is the fitted classifier before the current outcome is known, \(\widehat y_t\) is its class prediction, and \(L_t\in\{0,1\}\) is the bounded loss. A refit after observing \(y_t\) can affect date \(t+1\) and later, but it cannot alter \(\widehat y_t\).

In [1]:
from pathlib import Path
import sys
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
from scipy.stats import norm
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
import RiskLabAI
from RiskLabAI.backtest.adaptation import prequential_adaptation
from RiskLabAI.data.labeling import cusum_filter_events_dynamic_threshold, symmetric_cusum_filter
from RiskLabAI.features.structural_breaks.pelt_change_points import pelt_change_points
from river.drift import ADWIN
assert RiskLabAI.__version__ == '3.2.0'
warnings.filterwarnings('ignore', message='New behaviour in v1.1.5')
DATA_PATH = ensure_dataset(BOOK_ROOT, 'fred_nasdaq')
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '13_regime_adaptation'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '13_regime_adaptation'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '13_regime_adaptation'
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return '0'
        if number.is_integer():
            return f'{int(number):,}'
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f'{number:.6g}'
        return f'{number:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)

def load_public_data():
    assert DATA_PATH.is_file()
    raw = pd.read_csv(DATA_PATH)
    raw['Date'] = pd.to_datetime(raw['observation_date'])
    raw['NASDAQ Composite'] = pd.to_numeric(raw['NASDAQCOM'], errors='coerce')
    data = raw.dropna(subset=['NASDAQ Composite']).sort_values('Date').reset_index(drop=True)
    data['Daily log return (%)'] = 100 * np.log(data['NASDAQ Composite'] / data['NASDAQ Composite'].shift(1))
    data['Log index level'] = 100 * np.log(data['NASDAQ Composite'])
    return data

def prepare_sequential_data(data, min_samples=756):
    frame = data[['Date', 'Daily log return (%)']].copy()
    lagged = frame['Daily log return (%)'].shift(1)
    frame['Previous return (%)'] = lagged
    frame['Previous absolute return (%)'] = lagged.abs()
    frame['Trailing 5 day mean (%)'] = lagged.rolling(5).mean()
    frame['Trailing 21 day volatility (%)'] = lagged.rolling(21).std(ddof=0)
    frame['Trailing 63 day volatility (%)'] = lagged.rolling(63).std(ddof=0)
    frame = frame.dropna().reset_index(drop=True)
    threshold = float(frame.loc[:min_samples - 1, 'Daily log return (%)'].abs().quantile(0.8))
    frame['Large move'] = (frame['Daily log return (%)'].abs() > threshold).astype(int)
    return (frame, threshold)

def feature_columns():
    return ['Previous return (%)', 'Previous absolute return (%)', 'Trailing 5 day mean (%)', 'Trailing 21 day volatility (%)', 'Trailing 63 day volatility (%)']

def make_estimator():
    return Pipeline([('standardization', StandardScaler()), ('logistic regression', LogisticRegression(C=1, class_weight='balanced', random_state=314159, max_iter=1000))])

def zero_one_loss(actual, prediction):
    return float(actual != prediction)

def run_adaptive(features, targets, delta=0.01, min_samples=756, window_size=756):
    return prequential_adaptation(features, targets, make_estimator(), loss_function=zero_one_loss, window_size=window_size, min_samples=min_samples, delta=delta)

def run_static(features, targets, min_samples=756):
    predictions = np.full(len(targets), np.nan)
    losses = np.full(len(targets), np.nan)
    model = clone(make_estimator()).fit(features[:min_samples], targets[:min_samples])
    predictions[min_samples:] = model.predict(features[min_samples:])
    losses[min_samples:] = (predictions[min_samples:] != targets[min_samples:]).astype(float)
    return {'predictions': predictions, 'losses': losses, 'refit_indices': np.array([min_samples - 1], dtype=int)}

def run_periodic(features, targets, min_samples=756, window_size=756, interval=252):
    predictions = np.full(len(targets), np.nan)
    losses = np.full(len(targets), np.nan)
    refits = [min_samples - 1]
    model = clone(make_estimator()).fit(features[:min_samples], targets[:min_samples])
    for index in range(min_samples, len(targets)):
        predictions[index] = model.predict(features[index:index + 1])[0]
        losses[index] = float(predictions[index] != targets[index])
        if (index + 1 - min_samples) % interval == 0:
            begin = max(0, index + 1 - window_size)
            model = clone(make_estimator()).fit(features[begin:index + 1], targets[begin:index + 1])
            refits.append(index)
    return {'predictions': predictions, 'losses': losses, 'refit_indices': np.array(refits, dtype=int)}


In [2]:
data = load_public_data()
model_frame, large_move_threshold = prepare_sequential_data(data)
min_samples = 756
window_size = 756
interval = 252
features = model_frame[feature_columns()].to_numpy()
targets = model_frame["Large move"].to_numpy()

static = run_static(features, targets)
periodic = run_periodic(features, targets, interval=interval)
adaptive = run_adaptive(features, targets, delta=0.01)
evaluation_dates = model_frame.loc[min_samples:, "Date"].reset_index(drop=True)

policy_definitions = pd.DataFrame([
    {"Policy": "Static", "Refit rule": "No refit after the initial sample", "Rows used at a refit": 756, "Future information used": "No"},
    {"Policy": "Fixed schedule", "Refit rule": "Every 252 observed evaluation rows", "Rows used at a refit": 756, "Future information used": "No"},
    {"Policy": "ADWIN alarm", "Refit rule": "After an alarm in the bounded loss stream", "Rows used at a refit": 756, "Future information used": "No"},
])
save_table(policy_definitions, "policy_definitions", "Three predeclared refit policies compared on the same sequential task.", "tab:chapter14-policy-definitions", fit=True)

outputs = {"Static": static, "Fixed schedule": periodic, "ADWIN alarm": adaptive}
policy_rows = []
for name, output in outputs.items():
    predictions = output["predictions"][min_samples:]
    losses = output["losses"][min_samples:]
    actual = targets[min_samples:]
    true_positive = int(((predictions == 1) & (actual == 1)).sum())
    predicted_positive = int((predictions == 1).sum())
    actual_positive = int((actual == 1).sum())
    policy_rows.append({
        "Policy": name,
        "Evaluation observations": len(losses),
        "Mistakes": int(losses.sum()),
        "Mean loss": losses.mean(),
        "Accuracy": 1 - losses.mean(),
        "Predicted large moves": predicted_positive,
        "Actual large moves": actual_positive,
        "Large move precision": true_positive / predicted_positive,
        "Large move recall": true_positive / actual_positive,
        "Refits after initialization": len(output["refit_indices"]) - 1,
    })
policy_summary = pd.DataFrame(policy_rows)
save_table(policy_summary, "policy_summary", "Sequential loss and classification summaries for the three fixed policies.", "tab:chapter14-policy-summary", fit=True)

policy_data = pd.DataFrame({"Date": evaluation_dates, "Actual large move": targets[min_samples:]})
for name, output in outputs.items():
    policy_data[f"{name} prediction"] = output["predictions"][min_samples:]
    policy_data[f"{name} loss"] = output["losses"][min_samples:]
    policy_data[f"{name} rolling 126 day mean loss"] = pd.Series(output["losses"][min_samples:]).rolling(126, min_periods=30).mean().to_numpy()
    policy_data[f"{name} cumulative mistakes"] = np.cumsum(output["losses"][min_samples:])

rolling_columns = ["Date", *[f"{name} rolling 126 day mean loss" for name in outputs]]
policy_data[rolling_columns].to_csv(FIGURE_DATA / "rolling_policy_loss.csv", index=False)
figure, axis = plt.subplots(figsize=(11.2, 5.5), constrained_layout=True)
for name, color in [("Static", PALETTE["amber"]), ("Fixed schedule", PALETTE["green"]), ("ADWIN alarm", PALETTE["cyan"])]:
    axis.plot(policy_data["Date"], policy_data[f"{name} rolling 126 day mean loss"], color=color, linewidth=1.35, label=name)
axis.set_xlabel("Date")
axis.set_ylabel("Rolling mean loss")
axis.set_title("Rolling Sequential Loss Under Three Refit Policies")
axis.legend()
finish_axis(axis)
save_figure(figure, "rolling_policy_loss")
plt.close(figure)

cumulative_columns = ["Date", *[f"{name} cumulative mistakes" for name in outputs]]
policy_data[cumulative_columns].to_csv(FIGURE_DATA / "cumulative_mistakes.csv", index=False)
figure, axis = plt.subplots(figsize=(11.2, 5.5), constrained_layout=True)
for name, color in [("Static", PALETTE["amber"]), ("Fixed schedule", PALETTE["green"]), ("ADWIN alarm", PALETTE["cyan"])]:
    axis.plot(policy_data["Date"], policy_data[f"{name} cumulative mistakes"], color=color, linewidth=1.55, label=name)
axis.set_xlabel("Date")
axis.set_ylabel("Cumulative mistakes")
axis.set_title("Cumulative Prediction Mistakes")
axis.legend()
finish_axis(axis)
save_figure(figure, "cumulative_mistakes")
plt.close(figure)

,Policy,Refit rule,Rows used at a refit,Future information used
0,Static,No refit after the initial sample,756,No
1,Fixed schedule,Every 252 observed evaluation rows,756,No
2,ADWIN alarm,After an alarm in the bounded loss stream,756,No


,Policy,Evaluation observations,Mistakes,Mean loss,Accuracy,Predicted large moves,Actual large moves,Large move precision,Large move recall,Refits after initialization
0,Static,"7,235","3,018",0.417139,0.582861,"5,034","2,742",0.472586,0.867615,0
1,Fixed schedule,"7,235","2,435",0.336558,0.663442,"2,841","2,742",0.55403,0.574034,28
2,ADWIN alarm,"7,235","2,394",0.330891,0.669109,"3,080","2,742",0.556494,0.625091,9


## Comparing three adaptation schedules

For a trailing window of \(w\) observed losses, the rolling mean loss is

$$
\overline L_t^{(w)}
=\frac{1}{w}\sum_{s=t-w+1}^{t}L_s.
$$

Here, \(w=126\) trading observations, \(L_s\) is the zero or one mistake on date \(s\), and \(\overline L_t^{(w)}\) is the recent mistake rate. The static policy never changes its initial fit. The fixed schedule refits every 252 observed evaluation rows. The alarm policy refits only after ADWIN signals a change. All three policies use the same features, target, initial sample, estimator, and loss.

In [3]:
block_edges = [1996, 2001, 2006, 2011, 2016, 2021, 2025]
block_labels = ["1997 to 2001", "2002 to 2006", "2007 to 2011", "2012 to 2016", "2017 to 2021", "2022 to 2025"]
years = policy_data["Date"].dt.year
policy_data["Calendar block"] = pd.cut(years, bins=block_edges, labels=block_labels, include_lowest=True)
block_rows = []
for block in block_labels:
    selected = policy_data["Calendar block"] == block
    for name in outputs:
        block_rows.append({
            "Calendar block": block,
            "Policy": name,
            "Observations": int(selected.sum()),
            "Mistakes": int(policy_data.loc[selected, f"{name} loss"].sum()),
            "Mean loss": policy_data.loc[selected, f"{name} loss"].mean(),
        })
block_table = pd.DataFrame(block_rows)
save_table(block_table, "policy_calendar_blocks", "Sequential loss by fixed calendar block and refit policy.", "tab:chapter14-policy-calendar-blocks", fit=True)

refit_rows = []
for name, output in outputs.items():
    for number, index in enumerate(output["refit_indices"], start=1):
        refit_rows.append({
            "Policy": name,
            "Refit number": number,
            "Observation date": model_frame.loc[index, "Date"],
            "Reason": "Initial sample" if number == 1 else "Fixed schedule" if name == "Fixed schedule" else "ADWIN alarm",
        })
refit_table = pd.DataFrame(refit_rows)
save_table(refit_table, "policy_refit_dates", "Dates at which each policy fits or refits its classifier.", "tab:chapter14-policy-refit-dates", fit=True)

refit_figure = refit_table.loc[refit_table["Refit number"] > 1, ["Policy", "Observation date", "Reason"]].copy()
refit_figure.to_csv(FIGURE_DATA / "policy_refit_timeline.csv", index=False)
positions = {"Fixed schedule": 0, "ADWIN alarm": 1}
colors = {"Fixed schedule": PALETTE["green"], "ADWIN alarm": PALETTE["rose"]}
figure, axis = plt.subplots(figsize=(11.0, 4.2), constrained_layout=True)
for name, position in positions.items():
    dates = refit_figure.loc[refit_figure["Policy"] == name, "Observation date"]
    axis.vlines(dates, position - 0.28, position + 0.28, color=colors[name], linewidth=1.4)
axis.set_yticks(list(positions.values()), list(positions.keys()))
axis.set_xlabel("Date")
axis.set_title("Fixed Schedule and ADWIN Refit Dates")
axis.set_ylim(-0.65, 1.65)
finish_axis(axis, grid=False)
save_figure(figure, "policy_refit_timeline")
plt.close(figure)

order_checks = pd.DataFrame([
    {"Check": "All policies evaluate the same number of dates", "Result": len({len(output["losses"][min_samples:]) for output in outputs.values()}) == 1},
    {"Check": "All evaluation predictions are finite", "Result": all(np.isfinite(output["predictions"][min_samples:]).all() for output in outputs.values())},
    {"Check": "All losses are zero or one", "Result": all(np.isin(output["losses"][min_samples:], [0, 1]).all() for output in outputs.values())},
    {"Check": "Fixed schedule gaps are 252 observations", "Result": bool(np.all(np.diff(periodic["refit_indices"]) == 252))},
    {"Check": "ADWIN alarm dates equal later refit dates", "Result": bool(np.array_equal(adaptive["alarm_indices"], adaptive["refit_indices"][1:]))},
    {"Check": "Initial fit ends before the first evaluation date", "Result": model_frame.loc[min_samples - 1, "Date"] < model_frame.loc[min_samples, "Date"]},
])
save_table(order_checks, "policy_order_checks", "Mechanical checks for the common sequential comparison.", "tab:chapter14-policy-order-checks", fit=True)

,Calendar block,Policy,Observations,Mistakes,Mean loss
0,1997 to 2001,Static,"1,195",464,0.388285
1,1997 to 2001,Fixed schedule,"1,195",465,0.389121
2,1997 to 2001,ADWIN alarm,"1,195",465,0.389121
3,2002 to 2006,Static,"1,259",523,0.415409
4,2002 to 2006,Fixed schedule,"1,259",451,0.358221
5,2002 to 2006,ADWIN alarm,"1,259",406,0.322478
6,2007 to 2011,Static,"1,260",569,0.451587
7,2007 to 2011,Fixed schedule,"1,260",452,0.35873
8,2007 to 2011,ADWIN alarm,"1,260",448,0.355556
9,2012 to 2016,Static,"1,258",532,0.422893


,Policy,Refit number,Observation date,Reason
0,Static,1,1997-04-01,Initial sample
1,Fixed schedule,1,1997-04-01,Initial sample
2,Fixed schedule,2,1998-03-31,Fixed schedule
3,Fixed schedule,3,1999-03-31,Fixed schedule
4,Fixed schedule,4,2000-03-29,Fixed schedule
5,Fixed schedule,5,2001-03-29,Fixed schedule
6,Fixed schedule,6,2002-04-05,Fixed schedule
7,Fixed schedule,7,2003-04-04,Fixed schedule
8,Fixed schedule,8,2004-04-05,Fixed schedule
9,Fixed schedule,9,2005-04-06,Fixed schedule


,Check,Result
0,All policies evaluate the same number of dates,Pass
1,All evaluation predictions are finite,Pass
2,All losses are zero or one,Pass
3,Fixed schedule gaps are 252 observations,Pass
4,ADWIN alarm dates equal later refit dates,Pass
5,Initial fit ends before the first evaluation date,Pass


,Check,Result
0,All policies evaluate the same number of dates,Pass
1,All evaluation predictions are finite,Pass
2,All losses are zero or one,Pass
3,Fixed schedule gaps are 252 observations,Pass
4,ADWIN alarm dates equal later refit dates,Pass
5,Initial fit ends before the first evaluation date,Pass


## Fixed Sensitivity Comparison

The ADWIN confidence parameter \(\delta\) controls the probability level used by its change test. This notebook keeps \(\delta=0.01\) as the default and reports two predeclared alternatives, \(0.002\) and \(0.05\). Every other feature, target, estimator, initial sample, and rolling refit window remains unchanged. The sensitivity calculation describes how this one setting changes the observed alarms and loss in this sample; it is not used to replace the default after seeing the results.

In [4]:
sensitivity_rows = []
for delta in [0.002, 0.01, 0.05]:
    result = run_adaptive(features, targets, delta=delta)
    sensitivity_rows.append({
        "ADWIN confidence parameter": delta,
        "Alarms": len(result["alarm_indices"]),
        "Refits after initialization": len(result["refit_indices"]) - 1,
        "Mistakes": int(np.nansum(result["losses"])),
        "Mean loss": np.nanmean(result["losses"]),
        "Accuracy": 1 - np.nanmean(result["losses"]),
    })
sensitivity = pd.DataFrame(sensitivity_rows)
save_table(sensitivity, "delta_sensitivity", "Predeclared sensitivity of alarms and loss to the ADWIN confidence parameter.", "tab:chapter14-delta-sensitivity")
sensitivity.to_csv(FIGURE_DATA / "delta_sensitivity.csv", index=False)

figure, axes = plt.subplots(1, 2, figsize=(10.8, 4.8), constrained_layout=True)
labels = [f"{value:g}" for value in sensitivity["ADWIN confidence parameter"]]
axes[0].bar(labels, sensitivity["Alarms"], color=[PALETTE["violet"], PALETTE["cyan"], PALETTE["rose"]])
axes[0].set_xlabel("ADWIN confidence parameter")
axes[0].set_ylabel("Alarm count")
axes[0].set_title("Alarm Counts")
axes[1].plot(labels, sensitivity["Mean loss"], color=PALETTE["amber"], marker="o", linewidth=1.8)
axes[1].set_xlabel("ADWIN confidence parameter")
axes[1].set_ylabel("Mean loss")
axes[1].set_title("Sequential Mean Loss")
for axis in axes:
    finish_axis(axis)
save_figure(figure, "delta_sensitivity")
plt.close(figure)

,ADWIN confidence parameter,Alarms,Refits after initialization,Mistakes,Mean loss,Accuracy
0,0.002,7,7,"2,583",0.357015,0.642985
1,0.01,9,9,"2,394",0.330891,0.669109
2,0.05,11,11,"2,394",0.330891,0.669109


## What the Comparison Can and Cannot Show

The fixed public sample shows that the three predeclared policies can produce different loss paths and refit counts. It does not establish that one policy will remain better in another asset, period, target, cost structure, or model family. The comparison contains no trading position, return, transaction cost, or causal intervention. The observed loss differences are descriptive results for one sequential classification task.

In [5]:
limits = pd.DataFrame([
    {"Question": "What is compared?", "Permitted conclusion": "Prediction mistakes under three fixed refit policies", "Excluded conclusion": "Investment performance"},
    {"Question": "What does an ADWIN alarm mean?", "Permitted conclusion": "Evidence of a change in the bounded loss stream", "Excluded conclusion": "Proof of an economic cause"},
    {"Question": "What does a refit do?", "Permitted conclusion": "Changes later predictions after observed data are added", "Excluded conclusion": "Changes the prediction on the alarm date"},
    {"Question": "What does the public sample establish?", "Permitted conclusion": "A reproducible description of this task and period", "Excluded conclusion": "Universal forecasting superiority"},
    {"Question": "How is sensitivity used?", "Permitted conclusion": "Reports fixed alternative confidence parameters", "Excluded conclusion": "Selects the best setting after observing outcomes"},
])
save_table(limits, "policy_interpretation_limits", "Interpretation boundaries for the sequential policy comparison.", "tab:chapter14-policy-interpretation-limits", fit=True)

,Question,Permitted conclusion,Excluded conclusion
0,What is compared?,Prediction mistakes under three fixed refit po...,Investment performance
1,What does an ADWIN alarm mean?,Evidence of a change in the bounded loss stream,Proof of an economic cause
2,What does a refit do?,Changes later predictions after observed data ...,Changes the prediction on the alarm date
3,What does the public sample establish?,A reproducible description of this task and pe...,Universal forecasting superiority
4,How is sensitivity used?,Reports fixed alternative confidence parameters,Selects the best setting after observing outcomes


,Question,Permitted conclusion,Excluded conclusion
0,What is compared?,Prediction mistakes under three fixed refit po...,Investment performance
1,What does an ADWIN alarm mean?,Evidence of a change in the bounded loss stream,Proof of an economic cause
2,What does a refit do?,Changes later predictions after observed data ...,Changes the prediction on the alarm date
3,What does the public sample establish?,A reproducible description of this task and pe...,Universal forecasting superiority
4,How is sensitivity used?,Reports fixed alternative confidence parameters,Selects the best setting after observing outcomes
